### This script:
### 1. Loads the Step 3 feature matrix
### 2. Splits into train/test sets (stratified, 80/20)
### 3. Trains three models: Logistic Regression, Random Forest, XGBoost
### 4. Evaluates each with 5-fold cross-validation on the training set
### 5. Evaluates each on the held-out test set
### 6. Produces a comparison table of all models

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")
 
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import (roc_auc_score, average_precision_score,
                                     f1_score, classification_report,
                                     ConfusionMatrixDisplay)
import xgboost as xgb
import matplotlib
matplotlib.use("Agg")   # non-interactive backend for saving figures
import matplotlib.pyplot as plt

In [ ]:
INPUT_PATH = r"C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs\step3_full.csv"
OUTPUT_DIR = r"C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs"
LABEL_COL  = "dementia_binary"
ID_COL     = "NACCID"
RANDOM_STATE = 42
 
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"Loaded: {df.shape}")

In [ ]:
# 1. Prepare feature matrix and target
 

feature_cols = [c for c in df.columns if c not in [ID_COL, LABEL_COL]]
X = df[feature_cols].values
y = df[LABEL_COL].values
 
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Class balance: {pd.Series(y).value_counts(normalize=True).round(3).to_dict()}")
 

In [ ]:
# 2. Train / test split
#
# Stratified 80/20 split. Stratification preserves the 54.7/45.3 class balance
# in both train and test sets. Since we already reduced to one row per subject
# in Step 2, there is no subject-level leakage risk with a random row split.
 

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
 
print(f"Train size: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test size:  {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Train class balance: {pd.Series(y_train).value_counts(normalize=True).round(3).to_dict()}")
print(f"Test class balance:  {pd.Series(y_test).value_counts(normalize=True).round(3).to_dict()}")